# 0.0 - Imports

In [1]:
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# 0.1 - Helper Functions

## 0.1.2 - Path

In [2]:
# Data file path constants
DATA_DIR = "./data"
CUSTOMERS_FILE = "../data/olist_customers_dataset.csv"
GEOLOCATION_FILE = "../data/olist_geolocation_dataset.csv"
ORDER_ITEMS_FILE = "../data/olist_order_items_dataset.csv"
ORDER_PAYMENTS_FILE = "../data/olist_order_payments_dataset.csv"
ORDER_REVIEWS_FILE = "../data/olist_order_reviews_dataset.csv"
ORDERS_FILE = "../data/olist_orders_dataset.csv"
PRODUCTS_FILE = "../data/olist_products_dataset.csv"
SELLERS_FILE = "../data/olist_sellers_dataset.csv"
PRODUCT_CATEGORY_TRANSLATION_FILE = (
    "../data/product_category_name_translation.csv"
)

In [3]:
nltk.download("stopwords")
pt_stop_words = stopwords.words("portuguese")

[nltk_data] Downloading package stopwords to /home/elias/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 0.2 - Load Data

In [4]:
# Load datasets
orders = pd.read_csv(ORDERS_FILE)
items = pd.read_csv(ORDER_ITEMS_FILE)
products = pd.read_csv(PRODUCTS_FILE)
reviews = pd.read_csv(ORDER_REVIEWS_FILE)

# Merge datasets
df_raw = orders.merge(items, on="order_id", how="left")
df_raw = df_raw.merge(products, on="product_id", how="left")
df_raw = df_raw.merge(reviews, on="order_id", how="left")

df_raw.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,500.0,19.0,8.0,13.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,400.0,19.0,13.0,19.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,420.0,24.0,19.0,21.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,450.0,30.0,10.0,20.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,250.0,51.0,15.0,15.0,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


## 0.3 - Clean Data

In [5]:
df1 = df_raw.copy()

In [6]:
# Remove duplicates
df1 = df1.drop_duplicates()

# Handle missing values
df1 = df1.dropna(subset=["review_comment_message", "review_score"])

# Select relevant columns
df1 = df1[["review_score", "review_comment_message"]]
df1["sentimento"] = df1["review_score"].apply(lambda x: 1 if x >= 4 else 0)

df1["sentimento"].value_counts()

sentimento
1    29514
0    18652
Name: count, dtype: int64

# 1.0 - Sentiment Analysis

In [7]:
df2 = df1.copy()

In [8]:
vec = TfidfVectorizer(
    # stop_words=pt_stop_words,
    ngram_range=(1, 4),
    max_df=0.6,
    min_df=5,
)
tfidf_matrix = vec.fit_transform(df2["review_comment_message"])

# Transformando em DataFrame para visualizar
features = vec.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=features)

mean_pos = df_tfidf[df2["sentimento"].values == 1].mean()
mean_neg = df_tfidf[df2["sentimento"].values == 0].mean()

analise_sentimento = pd.DataFrame(
    {"positivo_score": mean_pos, "negativo_score": mean_neg}
)

analise_sentimento["diff"] = (
    analise_sentimento["positivo_score"] - analise_sentimento["negativo_score"]
)

analise_sentimento = analise_sentimento[
    (analise_sentimento.index.str.contains(" "))
    | (~analise_sentimento.index.isin(pt_stop_words))
]

# Ordena pelos termos mais longos primeiro
termos_ordenados = sorted(analise_sentimento.index, key=len, reverse=True)
termos_finais = []

for termo in termos_ordenados:
    # Se o termo não for uma sub-parte de algo que já adicionamos, ele é único
    if not any(termo in t_final for t_final in termos_finais):
        termos_finais.append(termo)

In [9]:
matriz_positiva = analise_sentimento[analise_sentimento["diff"] > 0].copy()
matriz_positiva = matriz_positiva.loc[
    matriz_positiva.index.isin(termos_finais)
]
matriz_positiva = matriz_positiva.sort_values(by="diff", ascending=False)
matriz_positiva = matriz_positiva["positivo_score"]
matriz_positiva.head(15)

chegou antes do prazo         0.007151
entregue antes do prazo       0.004611
bem antes do prazo            0.004554
entrega antes do prazo        0.003302
chegou bem antes do           0.002810
produto chegou antes do       0.002916
produto entregue antes do     0.002298
antes do prazo previsto       0.002153
produto de ótima qualidade    0.002021
recebi antes do prazo         0.001797
muito antes do prazo          0.001639
antes do prazo produto        0.001596
produto de boa qualidade      0.001693
antes da data prevista        0.001543
antes do prazo recomendo      0.001476
Name: positivo_score, dtype: float64

In [10]:
matriz_negativa = analise_sentimento[analise_sentimento["diff"] < 0].copy()
matriz_negativa = matriz_negativa.loc[
    matriz_negativa.index.isin(termos_finais)
]
matriz_negativa = matriz_negativa.sort_values(by="diff", ascending=True)
matriz_negativa = matriz_negativa["negativo_score"]
matriz_negativa.head(15)

ainda não recebi produto        0.002810
produto não foi entregue        0.002369
não recebi meu produto          0.001944
meu dinheiro de volta           0.001309
ainda não foi entregue          0.001290
até agora não recebi            0.001141
até momento não recebi          0.001058
quero meu dinheiro de           0.000859
produto ainda não foi           0.000834
ainda não recebi meu            0.000827
produto veio com defeito        0.000863
produto ainda não chegou        0.000803
não recebi produto ainda        0.000782
produto de péssima qualidade    0.000684
ainda nao recebi produto        0.000603
Name: negativo_score, dtype: float64

In [13]:
# Create output directory if it doesn't exist
output_dir = Path("../data/reviews")
output_dir.mkdir(parents=True, exist_ok=True)

# Save results
matriz_positiva.to_csv(output_dir / "matriz_positiva.csv", index=True)
matriz_negativa.to_csv(output_dir / "matriz_negativa.csv", index=True)